# 01 — Train/Test Split and Sampling

**Owner:** Data Preparation & Pipeline

**Purpose:** create the train and test sets that the whole team uses. This happens **before any cleaning or exploration**, so that nothing we learn from the test set can leak into our decisions.

**Outputs**
- `data/train.csv`: 150,000 loans issued 2007–2015 (stratified sample)
- `data/test.csv`: 50,000 loans issued in 2016 (stratified sample)

> ⚠️ Teammates: do **all** EDA on `train.csv` only. Do not open `test.csv` until final evaluation.

## Why split by time instead of randomly?

The model's real job is: *a lender trains on past loans, then scores new applications.* A time-based split reproduces that situation.

- **Train = loans issued 2007–2015**, **test = loans issued in 2016.** The test set plays the role of "future" applicants the model has never seen.
- A random split would mix 2016 loans into training, so the model would partly learn from the same period it is tested on. That gives an **over-optimistic** score.
- It also lets us check for **drift**: the default rate rises from about 20% (2015) to 23% (2016). If the model still performs well on 2016, that is real evidence it generalizes.

## Why exclude 2017–2018?

According to the dataset documentation, it contains **only finished loans** (fully paid = 0, charged off = 1). Loans still being repaid when the data was collected were removed.

Lending Club loans run 36 or 60 months, so many 2017–2018 loans had not finished yet. The ones that had finished are mostly early payoffs, so these years show an **artificially low** default rate (2018: 15.8%). Including them would teach the model a false pattern.

## Why sample?

The full dataset has 1.35M rows. Cross-validating 4–5 model families, including SVM and kNN, on that size would not fit in our 8-day timeline. A **stratified** sample keeps the same default rate as the full period, so the class balance is preserved.

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

RAW_PATH = "/Users/unaqian/Downloads/ML_lend_GroupProject/LC_loans_granting_model_dataset.csv"
RANDOM_STATE = 42          # fixed seed so every teammate gets the same sample
N_TRAIN, N_TEST = 150_000, 50_000

df = pd.read_csv(RAW_PATH, low_memory=False)
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
print(df.shape)

(1347681, 15)


## Loans and default rate by issue year

In [ ]:
by_year = df.groupby(df["issue_d"].dt.year)["Default"].agg(n_loans="size", default_rate="mean")
by_year.round(3)

,n_loans,default_rate
issue_d,,
2007,599,0.264
2008,2393,0.207
2009,5281,0.137
2010,12537,0.140
2011,21721,0.152
2012,53367,0.162
2013,134804,0.156
2014,223102,0.184
2015,375543,0.202


## Split by time, then sample within each period

In [ ]:
train_pool = df[df["issue_d"].dt.year <= 2015]
test_pool  = df[df["issue_d"].dt.year == 2016]
print(f"Train pool (2007–2015): {len(train_pool):,} loans")
print(f"Test pool  (2016):      {len(test_pool):,} loans")
print(f"Excluded   (2017–2018): {len(df) - len(train_pool) - len(test_pool):,} loans")

def stratified_sample(pool, n):
    sample, _ = train_test_split(pool, train_size=n, stratify=pool["Default"],
                                 random_state=RANDOM_STATE)
    return sample.sort_values("issue_d").reset_index(drop=True)

train = stratified_sample(train_pool, N_TRAIN)
test  = stratified_sample(test_pool, N_TEST)

Train pool (2007–2015): 829,347 loans
Test pool  (2016):      293,057 loans
Excluded   (2017–2018): 225,277 loans


## Sanity checks: sampling preserved the default rate

In [ ]:
summary = pd.DataFrame({
    "rows": [len(train_pool), len(train), len(test_pool), len(test)],
    "default_rate": [train_pool.Default.mean(), train.Default.mean(),
                     test_pool.Default.mean(), test.Default.mean()],
}, index=["train pool", "train sample", "test pool", "test sample"])
assert set(train.id).isdisjoint(test.id), "Overlap between train and test!"
summary.round(4)

,rows,default_rate
train pool,829347,0.1846
train sample,150000,0.1846
test pool,293057,0.2328
test sample,50000,0.2328


## Save

No columns are dropped or modified here. All cleaning happens inside the sklearn Pipeline (notebook 02), where it is fitted on the training set only.

In [ ]:
train.to_csv("../data/train.csv", index=False)
test.to_csv("../data/test.csv", index=False)
print("Saved train.csv and test.csv")

Saved train.csv and test.csv
